In [32]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import xml.etree.ElementTree as ET


In [33]:
# Replace with your ENTSO-E API key
API_KEY = "7613fbc5-d460-4741-a463-7a26610e89b3"

EIC_codes = [
    {"country": "IT", "bidding_zone": "10Y1001A1001A73I", "border_eics": ["10YSI-ELES-----O", "10Y1001A1001A63L", "10YFR-RTE------C", "10YCH-SWISSGRIDZ", "10YAT-APG------L", "10Y1001A1001A70O"]},
    # {"country": "DE", "bidding_zone": "10Y1001A1001A82H", "border_eics": ["10YPL-AREA-----S", "10YNL----------L", "10YFR-RTE------C", "10YDK-2--------M", "10YDK-1--------W", "10YCZ-CEPS-----N", "10YCH-SWISSGRIDZ", "10YBE----------2", "10YAT-APG------L", "10Y1001A1001A47J", "10YNO-2--------T"]},
    {"country": "PO", "bidding_zone": "10YPL-AREA-----S", "border_eics": ["10YSK-SEPS-----K", "10YLT-1001A0008Q", "10YDOM-CZ-DE-SKK", "10YCZ-CEPS-----N", "10Y1001A1001A63L", "10Y1001C--000182", "10Y1001A1001A82H", "10Y1001C--00003F", "10Y1001A1001A869", "10Y1001A1001A47J"]},
    {"country": "CH", "bidding_zone": "10YCH-SWISSGRIDZ", "border_eics": ["10YFR-RTE------C", "10YAT-APG------L", "10Y1001A1001A82H", "10Y1001A1001A73I", "10Y1001A1001A63L", "10Y1001A1001A68B"]},
    {"country": "FR", "bidding_zone": "10YFR-RTE------C", "border_eics": ["10Y1001A1001A73I", "10Y1001A1001A81J","17Y000000930814J", "10Y1001C--00098F", "10YES-REE------0", "10YGB----------A", "10Y1001A1001A82H", "10Y1001A1001A63L", "10YCH-SWISSGRIDZ", "10YBE----------2", "11Y0-0000-0265-K"]}

]

EIC_codes = [
]


# Time range: whole of 2024
start_date = datetime(2023, 1, 1, 0, 00)
end_date = datetime(2024, 1, 1, 0, 00)
period_start = start_date.strftime("%Y%m%d%H%M")
period_end = end_date.strftime("%Y%m%d%H%M")

In [34]:
# --- API REQUEST FUNCTION ---
def fetch_transparency_data(in_domain, out_domain):
    url = "https://web-api.tp.entsoe.eu/api"
    params = {
        "documentType": "A11",
        "in_Domain": in_domain,
        "out_Domain": out_domain,
        "periodStart": period_start,
        "periodEnd": period_end,
        "securityToken": API_KEY
    }
    try:
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {in_domain}->{out_domain}: {e}")
        with open("error_log.txt", "a") as f:
            f.write(f"{datetime.now()}: {e}\n")
        return None

In [35]:
def parse_xml(xml_content, direction_sign):
    if not xml_content:
        return None

    try:
        root = ET.fromstring(xml_content)
        namespace = {"ns": "urn:iec62325.351:tc57wg16:451-3:publicationdocument:7:0"}
        points = []

        for ts in root.findall(".//ns:TimeSeries", namespace):
            for period in ts.findall("ns:Period", namespace):
                start_time_str = period.find("ns:timeInterval/ns:start", namespace).text
                start_time = datetime.fromisoformat(start_time_str.replace('Z', '+00:00'))
                for point in period.findall("ns:Point", namespace):
                    position = int(point.find("ns:position", namespace).text)
                    quantity = float(point.find("ns:quantity", namespace).text)
                    timestamp = start_time + timedelta(minutes=15*(position-1))
                    points.append({
                        "timestamp": timestamp,
                        "flow_mw": quantity * direction_sign
                    })
        return pd.DataFrame(points)
    except Exception as e:
        print(f"XML parsing error: {e}")
        return None

In [36]:
# Process each country
for country in EIC_codes:
    country_code = country["country"]
    bidding_zone = country["bidding_zone"]
    
    # Create country df
    country_df = pd.DataFrame()

    print(f"\nProcessing {country_code}...")

    # Process each border
    for border_eic in country["border_eics"]:
        print(f"  Processing border {border_eic}...")

        # Fetch both directions (import and export)
        # out_domain and in_domain work opposite as what I initially thought:
        # out_domain=IT_North and in_domain=SI means the receiving end is IT and SI is the sending end
        for out_domain, in_domain, direction_sign in [
            (bidding_zone, border_eic, 1),  # Export from country (positive)
            (border_eic, bidding_zone, -1)     # Import to country (negative)
        ]:
            data = fetch_transparency_data(in_domain, out_domain)
            if data:
                df = parse_xml(data, direction_sign)
                
                if df is not None and len(df) > 0:
                    print(f"        {len(df)} records for {in_domain}->{out_domain}, resampling to hourly...")
                    # Resample to hourly instead of 15-minute intervals
                    df.set_index("timestamp", inplace=True)
                    # Resample to hourly by summing every 4 consecutive 15-min values
                    hourly_df = df.resample('H').sum().reset_index()
                    
                    # Add flow_mw values to country_df, aligning on timestamp
                    if country_df.empty:
                        country_df = hourly_df.copy()
                    else:
                        country_df = country_df.set_index("timestamp").add(
                            hourly_df.set_index("timestamp"), fill_value=0
                        ).reset_index()
                    # Some exchanges are obsolete and do not have data
                        
                    
    # Save to CSV
    filename = f"{country_code}_net_flow_2024.csv"
    country_df.to_csv(filename, index=False)
    print(f"Saved {filename} with {len(country_df)} records")
